# Seizure Prediction using Random Forest

In [16]:
"""
EEG Seizure Classification Pipeline
====================================
Input:  CHB-MIT .edf files, structured as a patient dictionary below.
Output: Random Forest classifier with per-patient leave-one-out evaluation.
        Reports sensitivity, specificity, and AUC — NOT raw accuracy.

Dependencies:
    pip install mne numpy scipy scikit-learn
"""

import mne
import numpy as np
from pathlib import Path
from scipy.signal import butter, filtfilt, iirnotch, welch
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.utils import resample
import warnings
warnings.filterwarnings('ignore')
mne.set_log_level('ERROR')


# =============================================================================
# CONFIGURATION
# =============================================================================

DATA_ROOT = Path("C:/Users/Mary Yap/HonoursResearch")

# normal_edf      = a file with NO seizure, used as the interictal source
# pre_ictal_edf   = a file where a seizure occurs; seizure_start_sec is within it
# seizure_start_sec = onset in seconds, relative to the START of pre_ictal_edf
PATIENTS = [
    {
        "id": "patient_01",
        "normal_edf":       "chb01/chb01_01.edf",
        "pre_ictal_edf":    "chb01/chb01_16.edf",
        "seizure_start_sec": 1015,
    },
    {
        "id": "patient_02",
        "normal_edf":       "chb02/chb02_01.edf",
        "pre_ictal_edf":    "chb02/chb02_16+.edf",
        "seizure_start_sec": 2972,
    },
    {
        "id": "patient_03",
        "normal_edf":       "chb03/chb03_06.edf",
        "pre_ictal_edf":    "chb03/chb03_02.edf",
        "seizure_start_sec": 731,
    },
    {
        "id": "patient_04",
        "normal_edf":       "chb05/chb05_01.edf",
        "pre_ictal_edf":    "chb05/chb05_13.edf",
        "seizure_start_sec": 1086,
    },
]

# Windowing parameters
WIN_SEC        = 10       # window length in seconds
OVERLAP        = 0.5      # 50% overlap between windows
PREICTAL_MIN   = 10       # minutes before seizure onset to label as preictal
ICTAL_BUFFER   = 30       # seconds after seizure onset to discard (avoid ictal leaking in)
INTERICTAL_MIN = 30       # minutes of interictal to use from the normal file

# Frequency bands for feature extraction
BANDS = {
    "delta": (0.5, 4),
    "theta": (4,   8),
    "alpha": (8,   13),
    "beta":  (13,  30),
    "gamma": (30,  50),
}

# =============================================================================
# STEP 1 — Load a single EDF file
# =============================================================================

def load_edf(relative_path):
    """
    Load one EDF file. Returns (data, sfreq, ch_names).
    data shape: (n_channels, n_samples)
    """
    full_path = DATA_ROOT / relative_path
    if not full_path.exists():
        raise FileNotFoundError(f"EDF not found: {full_path}")

    raw = mne.io.read_raw_edf(str(full_path), preload=True, verbose=False)
    data = raw.get_data()          # µV, shape (n_ch, n_samples)
    sfreq = raw.info['sfreq']
    ch_names = raw.ch_names

    print(f"  Loaded {relative_path}  |  "
          f"{data.shape[0]} ch  |  {sfreq:.0f} Hz  |  "
          f"{data.shape[1]/sfreq/60:.1f} min")

    return data, float(sfreq), ch_names


# =============================================================================
# STEP 2 — Preprocess a single window
# =============================================================================

def preprocess_window(win, sfreq):
    """
    Bandpass + notch filter, then z-score and clip.
    win shape: (n_channels, n_samples)
    Returns same shape.
    """
    # Bandpass 0.5–50 Hz
    b, a = butter(4, [0.5, 50.0], btype='bandpass', fs=sfreq)
    win = filtfilt(b, a, win, axis=-1)

    # Notch at 60 Hz (US power line; CHB-MIT recorded in Boston)
    b, a = iirnotch(60.0, 30.0, sfreq)
    win = filtfilt(b, a, win, axis=-1)

    # Z-score per channel, then clip extreme artifacts
    #mu    = win.mean(axis=-1, keepdims=True)
    #sigma = win.std(axis=-1,  keepdims=True) + 1e-8
    #win   = (win - mu) / sigma
    #win   = np.clip(win, -5.0, 5.0)

    return win


# =============================================================================
# STEP 3 — Extract features from one preprocessed window
# =============================================================================

def extract_features(win, sfreq):
    """
    Compute band power (absolute + relative) and time-domain stats per channel.
    win shape: (n_channels, n_samples)
    Returns 1-D feature vector.
    """
    feats = []
    nperseg = min(int(sfreq * 2), win.shape[-1])   # 2-second segments for Welch

    for ch_sig in win:
        freqs, psd = welch(ch_sig, fs=sfreq, nperseg=nperseg)
        total_power = np.trapz(psd, freqs) + 1e-12

        for lo, hi in BANDS.values():
            idx = (freqs >= lo) & (freqs <= hi)
            bp  = np.trapz(psd[idx], freqs[idx])
            feats.append(np.log1p(bp))          # log absolute band power
            feats.append(bp / total_power)       # relative band power

        # Time-domain features
        feats.append(np.var(ch_sig))                         # variance
        feats.append(np.mean(np.abs(np.diff(ch_sig))))       # line length
        feats.append(float(np.percentile(np.abs(ch_sig), 95))) # 95th pct amplitude

    return np.array(feats, dtype=np.float32)


# =============================================================================
# STEP 4 — Slice one EDF into windows and label them
# =============================================================================

def slice_and_label(data, sfreq, label, start_sec=0.0, end_sec=None):
    """
    Cut data[start_sec:end_sec] into overlapping windows.
    All windows receive the same integer label (0=interictal, 1=preictal).

    Returns:
        windows : list of np.ndarray (n_channels, win_samples)
        labels  : list of int
    """
    n_samples   = data.shape[1]
    win_samples = int(WIN_SEC * sfreq)
    step        = int(win_samples * (1 - OVERLAP))

    start_idx = int(start_sec * sfreq)
    end_idx   = int(end_sec   * sfreq) if end_sec is not None else n_samples
    end_idx   = min(end_idx, n_samples)

    windows, labels_out = [], []

    for s in range(start_idx, end_idx - win_samples + 1, step):
        e   = s + win_samples
        win = data[:, s:e]

        # Skip windows with flat or NaN data
        if not np.isfinite(win).all():
            continue
        if np.mean(np.std(win, axis=-1)) < 1e-6:
            continue

        win_clean = preprocess_window(win.copy(), sfreq)
        feat      = extract_features(win_clean, sfreq)

        windows.append(feat)
        labels_out.append(label)

    return windows, labels_out


# =============================================================================
# STEP 5 — Build dataset for one patient
# =============================================================================

def build_patient_dataset(patient):
    """
    For one patient dict, load both EDF files and extract labelled feature vectors.

    Returns:
        X : np.ndarray (n_windows, n_features)
        y : np.ndarray (n_windows,)   0=interictal, 1=preictal
    """
    pid = patient["id"]
    print(f"\n{'='*60}")
    print(f"Processing {pid}")
    print(f"{'='*60}")

    # --- Load files ---
    normal_data,   sfreq_n, _ = load_edf(patient["normal_edf"])
    preictal_data, sfreq_p, _ = load_edf(patient["pre_ictal_edf"])

    if sfreq_n != sfreq_p:
        raise ValueError(f"{pid}: sampling rates differ "
                         f"({sfreq_n} vs {sfreq_p}). Cannot mix files.")
    sfreq = sfreq_n

    seizure_onset = float(patient["seizure_start_sec"])

    # --- Interictal windows (from the normal file) ---
    normal_dur = normal_data.shape[1] / sfreq
    if INTERICTAL_MIN is not None:
        inter_end = min(float(INTERICTAL_MIN * 60), normal_dur)
    else:
        inter_end = normal_dur

    print(f"  Interictal: first {inter_end/60:.1f} min of normal file")
    X_inter, y_inter = slice_and_label(
        normal_data, sfreq, label=0,
        start_sec=0.0, end_sec=inter_end
    )

    # --- Preictal windows (30 min before seizure onset) ---
    preictal_start = max(0.0, seizure_onset - PREICTAL_MIN * 60)
    preictal_end   = seizure_onset - ICTAL_BUFFER   # leave a small buffer before onset

    if preictal_end <= preictal_start:
        raise ValueError(
            f"{pid}: seizure_start_sec={seizure_onset} is too early — "
            f"less than {PREICTAL_MIN} min into the file. "
            f"Use a different pre_ictal_edf or reduce PREICTAL_MIN."
        )

    print(f"  Preictal:  {preictal_start/60:.1f}–{preictal_end/60:.1f} min "
          f"of pre_ictal file  ({(preictal_end-preictal_start)/60:.1f} min window)")

    X_pre, y_pre = slice_and_label(
        preictal_data, sfreq, label=1,
        start_sec=preictal_start, end_sec=preictal_end
    )

    if len(X_inter) == 0 or len(X_pre) == 0:
        raise ValueError(f"{pid}: ended up with 0 windows in one class. "
                         f"Check your file paths and seizure_start_sec.")

    print(f"  Windows — interictal: {len(X_inter)}, preictal: {len(X_pre)}")

    X = np.vstack([X_inter, X_pre])
    y = np.array(y_inter + y_pre, dtype=int)

    return X, y


# =============================================================================
# STEP 6 — Balance classes (undersample majority)
# =============================================================================

def balance_classes(X, y, ratio=3):
    """
    Undersample the majority class to at most ratio * minority class size.
    ratio=3 means at most 3 interictal windows per 1 preictal window.
    """
    idx_0 = np.where(y == 0)[0]
    idx_1 = np.where(y == 1)[0]

    minority = min(len(idx_0), len(idx_1))
    majority = max(len(idx_0), len(idx_1))
    target   = min(majority, minority * ratio)

    if len(idx_0) > len(idx_1):
        idx_0 = resample(idx_0, n_samples=target, replace=False, random_state=42)
    else:
        idx_1 = resample(idx_1, n_samples=target, replace=False, random_state=42)

    idx_keep = np.concatenate([idx_0, idx_1])
    np.random.shuffle(idx_keep)
    return X[idx_keep], y[idx_keep]


# =============================================================================
# STEP 7 — Evaluate using leave-one-patient-out cross-validation
# =============================================================================
def temporal_smoothing(predictions, window_size=6, threshold=4):
    """
    Only labels a window as '1' if at least 'threshold' windows 
    out of the last 'window_size' were labeled '1'.
    """
    smoothed = np.zeros_like(predictions)
    for i in range(window_size, len(predictions)):
        # Look at the last 'window_size' predictions
        window = predictions[i-window_size : i]
        if np.sum(window) >= threshold:
            smoothed[i] = 1
        else:
            smoothed[i] = 0
    return smoothed
    
def run_pipeline(patients):
    """
    Main entry point.
    Builds features for all patients, then does leave-one-patient-out CV.
    Reports sensitivity, specificity, and AUC per fold.
    """
    print("\n" + "="*60)
    print("Building feature datasets for all patients...")
    print("="*60)

    all_X, all_y, all_pid = [], [], []

    for p in patients:
        try:
            X, y = build_patient_dataset(p)
            all_X.append(X)
            all_y.append(y)
            all_pid.extend([p["id"]] * len(y))
        except Exception as e:
            print(f"\n  ERROR for {p['id']}: {e}")
            print("  Skipping this patient.\n")

    if len(all_X) < 2:
        print("\nNeed at least 2 patients for leave-one-out CV.")
        print("If you only have 1 patient, use a simple train/test split instead.")
        return

    X_all   = np.vstack(all_X)
    y_all   = np.concatenate(all_y)
    pid_all = np.array(all_pid)

    patient_ids = sorted(set(pid_all))
    print(f"\nTotal windows: {len(y_all)} "
          f"(interictal={int((y_all==0).sum())}, preictal={int((y_all==1).sum())})")

    # -------------------------------------------------------------------------
    # Leave-one-patient-out cross-validation
    # -------------------------------------------------------------------------
    print("\n" + "="*60)
    print("Leave-one-patient-out cross-validation")
    print("="*60)

    fold_results = []

    for test_pid in patient_ids:
        train_mask = pid_all != test_pid
        test_mask  = pid_all == test_pid

        X_train, y_train = X_all[train_mask], y_all[train_mask]
        X_test,  y_test  = X_all[test_mask],  y_all[test_mask]

        # Balance training set only — never touch the test set
        X_train, y_train = balance_classes(X_train, y_train, ratio=3)

        if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
            print(f"\nFold {test_pid}: skipped (only one class present)")
            continue

        model = Pipeline([
            ('scaler', StandardScaler()),
            ('clf', RandomForestClassifier(
                n_estimators=300,
                class_weight='balanced',
                max_features='sqrt',
                min_samples_leaf=2,
                random_state=42,
                n_jobs=-1
            ))
        ])

        model.fit(X_train, y_train)

        # 1. Get the importances from the classifier
        importances = model.named_steps['clf'].feature_importances_
        
        # 2. Reconstruct the feature names (This is key!)
        feature_names = []
        for ch in range(23): # Assuming 23 channels
            for band in BANDS.keys():
                feature_names.append(f"Ch{ch}_{band}_AbsLog")
                feature_names.append(f"Ch{ch}_{band}_Rel")
            feature_names.append(f"Ch{ch}_Var")
            feature_names.append(f"Ch{ch}_LineLength")
            feature_names.append(f"Ch{ch}_95thPct")
        
        # 3. Pair them and sort
        feature_importance_dict = dict(zip(feature_names, importances))
        sorted_importance = sorted(feature_importance_dict.items(), key=lambda x: x[1], reverse=True)
        
        print(f"\nTop 10 Features for fold {test_pid}:")
        for name, val in sorted_importance[:10]:
            print(f"  {name}: {val:.4f}")
    
        y_pred_raw = model.predict(X_test)
        y_pred = temporal_smoothing(y_pred_raw, window_size=6, threshold=4)
        y_prob = model.predict_proba(X_test)[:, 1]

        cm = confusion_matrix(y_test, y_pred)

        # Safely extract TN, FP, FN, TP even if cm is not 2x2
        if cm.shape == (2, 2):
            tn, fp, fn, tp = cm.ravel()
        else:
            # Edge case: only one class predicted
            tn, fp, fn, tp = 0, 0, 0, 0
            print(f"  Warning: degenerate confusion matrix for {test_pid}")

        sensitivity  = tp / (tp + fn + 1e-8)   # recall on preictal
        specificity  = tn / (tn + fp + 1e-8)   # recall on interictal

        try:
            auc = roc_auc_score(y_test, y_prob)
        except ValueError:
            auc = float('nan')

        fold_results.append({
            "patient":     test_pid,
            "sensitivity": sensitivity,
            "specificity": specificity,
            "auc":         auc,
            "n_test":      len(y_test),
            "n_preictal":  int((y_test == 1).sum()),
        })

        print(f"\nTest patient: {test_pid}  ({len(y_test)} windows, "
              f"{int((y_test==1).sum())} preictal)")
        print(classification_report(y_test, y_pred,
              target_names=['interictal', 'preictal'], digits=3))
        print(f"  Sensitivity (preictal recall): {sensitivity:.3f}")
        print(f"  Specificity (interictal recall): {specificity:.3f}")
        print(f"  AUC:                            {auc:.3f}")

    # -------------------------------------------------------------------------
    # Summary across all folds
    # -------------------------------------------------------------------------
    if fold_results:
        print("\n" + "="*60)
        print("Summary across all folds")
        print("="*60)
        mean_sens = np.mean([r["sensitivity"] for r in fold_results])
        mean_spec = np.mean([r["specificity"] for r in fold_results])
        mean_auc  = np.mean([r["auc"]         for r in fold_results
                             if not np.isnan(r["auc"])])

        print(f"  Mean sensitivity : {mean_sens:.3f}")
        print(f"  Mean specificity : {mean_spec:.3f}")
        print(f"  Mean AUC         : {mean_auc:.3f}")
        print()
        print("  Per-patient results:")
        for r in fold_results:
            print(f"    {r['patient']:12s}  sens={r['sensitivity']:.3f}  "
                  f"spec={r['specificity']:.3f}  auc={r['auc']:.3f}")

        print("\nDone.")
        return fold_results

    else:
        print("\nNo valid folds produced. Check your data and paths.")
        return []


# =============================================================================
# ENTRY POINT
# =============================================================================

if __name__ == "__main__":
    results = run_pipeline(PATIENTS)



Building feature datasets for all patients...

Processing patient_01
  Loaded chb01/chb01_01.edf  |  23 ch  |  256 Hz  |  60.0 min
  Loaded chb01/chb01_16.edf  |  23 ch  |  256 Hz  |  60.0 min
  Interictal: first 30.0 min of normal file
  Preictal:  6.9–16.4 min of pre_ictal file  (9.5 min window)
  Windows — interictal: 359, preictal: 113

Processing patient_02
  Loaded chb02/chb02_01.edf  |  23 ch  |  256 Hz  |  60.0 min
  Loaded chb02/chb02_16+.edf  |  23 ch  |  256 Hz  |  60.0 min
  Interictal: first 30.0 min of normal file
  Preictal:  39.5–49.0 min of pre_ictal file  (9.5 min window)
  Windows — interictal: 359, preictal: 113

Processing patient_03
  Loaded chb03/chb03_06.edf  |  23 ch  |  256 Hz  |  60.0 min
  Loaded chb03/chb03_02.edf  |  23 ch  |  256 Hz  |  60.0 min
  Interictal: first 30.0 min of normal file
  Preictal:  2.2–11.7 min of pre_ictal file  (9.5 min window)
  Windows — interictal: 359, preictal: 113

Processing patient_04
  Loaded chb05/chb05_01.edf  |  23 ch  |

In [18]:
"""
Mean sensitivity: 0.631
Mean specificity : 0.978
Mean AUC         : 0.878

Per-patient results:
    patient_01    sens=0.814  spec=1.000  auc=0.979
    patient_02    sens=0.496  spec=0.978  auc=0.924
    patient_03    sens=0.283  spec=0.947  auc=0.642
    patient_04    sens=0.929  spec=0.989  auc=0.966
"""

'\nMean sensitivity: 0.631\nMean specificity : 0.978\nMean AUC         : 0.878\n\nPer-patient results:\n    patient_01    sens=0.814  spec=1.000  auc=0.979\n    patient_02    sens=0.496  spec=0.978  auc=0.924\n    patient_03    sens=0.283  spec=0.947  auc=0.642\n    patient_04    sens=0.929  spec=0.989  auc=0.966\n'

In [ ]:
============================================================
Summary across all folds
============================================================
  Mean sensitivity : 0.571
  Mean specificity : 0.634
  Mean AUC         : 0.789

  Per-patient results:
    patient_01    sens=0.619  spec=0.501  auc=0.623
    patient_02    sens=0.000  spec=0.955  auc=0.784
    patient_03    sens=0.929  spec=0.081  auc=0.810
    patient_04    sens=0.735  spec=1.000  auc=0.938

============================================================
Summary across all folds
============================================================
  Mean sensitivity : 0.518
  Mean specificity : 0.831
  Mean AUC         : 0.807

  Per-patient results:
    patient_01    sens=0.673  spec=0.914  auc=0.870
    patient_02    sens=0.248  spec=0.947  auc=0.872
    patient_03    sens=0.283  spec=0.914  auc=0.689
    patient_04    sens=0.867  spec=0.549  auc=0.797


